# Job Interview Guide Workshop — ML Specialist

**Interview result:** 15/15 (100%). No incorrect answers were identified.

**Purpose:** Since no weak topics were found in the multiple-choice assessment, this notebook provides practical mastery checks instead of inventing weaknesses. Inspired by the uploaded workshop notebooks: concept → code → experiment → interpretation → reflection.

**Learning objectives**
1. Identify features and targets and avoid data leakage.
2. Implement and evaluate regression with MSE and R².
3. Evaluate classification with log-loss, recall and precision.
4. Compare KNN and decision trees with reproducible pipelines.
5. Explain model selection trade-offs to a technical interviewer.

**Instructions:** Run cells in order. Complete each `TODO` before reading its sanity check. No external data or network access required.

## 0. Environment and reproducibility
The source workshops use Python, NumPy, pandas, scikit-learn, and notebook-based exploration. Set a random seed and use bundled scikit-learn datasets to keep this exercise reproducible.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score, log_loss, recall_score, precision_score, confusion_matrix
RANDOM_STATE = 42
print("NumPy:", np.__version__)
print("Ready for workshop")

NumPy: 2.5.1
Ready for workshop


## 1. Features, target, and leakage
**Concept:** Features (`X`) are the predictors; the target (`y`) is the quantity to predict. Fit learned preprocessing transformations on training data only. In scikit-learn, a `Pipeline` ensures that scaling is learned within each training fold during cross-validation.

**Exercise:** Load diabetes data and create an 80/20 split. Do not use the target as a feature.

In [2]:
diabetes = load_diabetes(as_frame=True, scaled=False)
df = diabetes.frame.copy()
print(df.head())
print("Rows, columns:", df.shape)
# TODO 1: Replace the following placeholders.
X = None  # e.g. df.drop(columns=[...])
y = None  # e.g. df[...]
# TODO 2: Create X_train, X_test, y_train, y_test with test_size=0.2 and random_state=RANDOM_STATE.
# X_train, X_test, y_train, y_test = ...

    age  sex   bmi     bp     s1     s2    s3   s4      s5    s6  target
0  59.0  2.0  32.1  101.0  157.0   93.2  38.0  4.0  4.8598  87.0   151.0
1  48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.8918  69.0    75.0
2  72.0  2.0  30.5   93.0  156.0   93.6  41.0  4.0  4.6728  85.0   141.0
3  24.0  1.0  25.3   84.0  198.0  131.4  40.0  5.0  4.8903  89.0   206.0
4  50.0  1.0  23.0  101.0  192.0  125.4  52.0  4.0  4.2905  80.0   135.0
Rows, columns: (442, 11)


In [3]:
# Sanity checks: run after completing the TODOs.
if X is not None and y is not None and 'X_train' in globals():
    assert 'target' not in X.columns
    assert len(X) == len(y) == 442
    assert len(X_train) + len(X_test) == 442
    assert set(X_train.index).isdisjoint(set(X_test.index))
    print("PASS: no target leakage and disjoint splits")
else:
    print("Complete TODOs in section 1 first.")

Complete TODOs in section 1 first.


## 2. Linear regression: predictions, MSE and R²
**Concept:** Linear regression predicts continuous values. MSE averages squared residuals, while R² compares predictive squared error with a mean-only baseline. A higher test R² is generally preferable when comparing models on the same evaluation data.

**Exercise:** Fit a model and calculate both metrics manually or using scikit-learn.

In [4]:
if 'X_train' in globals():
    reg = LinearRegression()
    # TODO 3: Fit reg on the training data and create y_pred on X_test.
    # reg.fit(...)
    # y_pred = ...
    # TODO 4: Compute MSE and R².
    # mse = ...
    # r2 = ...
    print("Training rows:", len(X_train))
else:
    print("Complete section 1 first.")

Complete section 1 first.


In [5]:
# Sanity check for Q5: prices are in thousands of dollars.
actual = np.array([200., 300., 400.])
predicted = np.array([210., 290., 420.])
assert np.isclose(mean_squared_error(actual, predicted), 200.0)
print("Reference MSE:", mean_squared_error(actual, predicted))
# TODO 5: Write a NumPy expression for MSE without calling sklearn.

Reference MSE: 200.0


## 3. Logistic regression: probability, threshold, and log-loss
**Concept:** Logistic regression models a binary outcome. Sigmoid maps a real-valued score to a probability. Binary cross-entropy penalizes confident incorrect probabilities. Choosing a threshold affects precision and recall.

**Exercise:** Calculate sigmoid and per-example log-loss for a positive observation.

In [6]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))
assert np.isclose(sigmoid(0), 0.5)
print("sigmoid(0):", sigmoid(0))
# TODO 6: Calculate log-loss for y=1 at p=0.90 and p=0.20.
# Use -np.log(p) or sklearn.metrics.log_loss with labels=[0,1].
# loss_a = ...
# loss_b = ...
# assert loss_a < loss_b

sigmoid(0): 0.5


## 4. KNN: scaling and hyperparameter tuning
**Concept:** Euclidean-distance KNN is sensitive to numerical scales. Tune `n_neighbors` on training data using cross-validation, with scaling inside the pipeline. Never choose K based on the test set.

**Exercise:** Build a small synthetic binary dataset, then compare k values.

In [7]:
X_cls, y_cls = make_classification(n_samples=400, n_features=6, n_informative=4,
                                    n_redundant=0, weights=[0.8, 0.2], random_state=RANDOM_STATE)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.25, stratify=y_cls, random_state=RANDOM_STATE)
knn_pipe = Pipeline([('scaler', StandardScaler()), ('classifier', KNeighborsClassifier())])
# TODO 7: Define GridSearchCV with parameter grid {'classifier__n_neighbors':[1,3,5,9,15]},
# cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE), scoring='f1'.
# Fit on Xc_train/yc_train, and print best_params_ and test recall.
assert set(np.unique(yc_train)) == {0, 1}
print("Train/test sizes:", len(yc_train), len(yc_test))

Train/test sizes: 300 100


## 5. Decision trees: leaf predictions
**Concept:** A squared-error regression tree predicts the average target value in a leaf. A classification tree predicts the most frequent class by default, and class probabilities reflect proportions of training examples in the leaf.

**Exercise:** Reproduce the interview leaf examples and inspect the results.

In [8]:
leaf_prices = np.array([200, 240, 260, 300])
leaf_labels = np.array([0]*15 + [1]*5)
# TODO 8: Compute the regression leaf prediction with np.mean.
# leaf_prediction = ...
# TODO 9: Compute probability of class 1 and the majority class.
# p_default = ...
# predicted_class = ...
assert np.isclose(np.mean(leaf_prices), 250)
assert np.isclose(np.mean(leaf_labels), 0.25)
print("Expected regression leaf mean:", np.mean(leaf_prices))

Expected regression leaf mean: 250.0


## 6. Scenario: imbalanced fraud detection
**Concept:** Accuracy can be misleading for rare events. Recall measures the fraction of actual positive cases found. Precision measures the fraction of predicted positives that are truly positive. Select metrics based on business costs.

**Exercise:** Compare two classifiers on a realistic imbalanced synthetic dataset. Use a held-out test set only for final reporting.

In [9]:
fraud_X, fraud_y = make_classification(n_samples=1000, n_features=8, n_informative=5,
    n_redundant=0, weights=[0.95, 0.05], flip_y=0, random_state=RANDOM_STATE)
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    fraud_X, fraud_y, test_size=0.25, stratify=fraud_y, random_state=RANDOM_STATE)
models = {
    'logistic': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    'tree': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
}
# TODO 10: Fit both models, predict on Xf_test, then print confusion_matrix,
# precision_score(zero_division=0) and recall_score for each.
print("Test fraud prevalence:", yf_test.mean())

Test fraud prevalence: 0.048


## 7. Interview reflection (write in your own words)
1. Why must a scaler be fitted on the training data rather than the complete dataset?
2. Explain MSE and R² to a nontechnical manager in two sentences.
3. When could higher recall be worth lower precision? Give a business example.
4. Explain why a very deep decision tree might perform worse on unseen data.
5. What is one concept you understand better after completing the code exercises?

**Exit ticket:** Choose one model from this notebook, explain why it fits its prediction target, identify a possible failure mode, and name a metric you would monitor in production.

**Next step:** Commit the completed notebook and add a brief README with environment requirements, execution instructions, and results.

---

## Contribution by Sukhchain Singh (9111541)

The following section is my contribution. The teammate notebook above is preserved in its original order.

# My LLM Workshop Contribution — Sukhchain Singh

**Student:** Sukhchain Singh  
**Student ID:** 9111541  
**Course:** Foundations of Machine Learning  
**Group:** Group 2  
**Contribution:** Study material review, individual 15-question quiz, results, and targeted workshop practice.

This section records Sukhchain Singh's part of the Job Interview Guide Workshop. The teammate's original notebook content is preserved after this section.


## My process, following the professor's instructions

### Step 1 — Collect the study materials

I used the course StudyGuide.txt and the five workshop areas listed in the professor's instructions:

1. DataStreamVisualization_Workshop
2. LinearRegressionArchitecture_Workshop
3. PerformanceMetricsClassification
4. KNearestNeighbors_Workshop
5. LogisticRegressionClassifier

I gathered the Jupyter notebooks from these workshop folders into StudyMaterials.zip. The archive contains nine .ipynb files across the five workshop areas; see [StudyMaterials.zip](StudyMaterials.zip). The study guide was attached in this LLM session. The workshop notebooks were reviewed from the GitHub repositories; the archive was prepared for the course workflow.

### Step 2 — Upload materials to an LLM

The professor's workflow asks students to start a new LLM conversation and upload both StudyGuide.txt and StudyMaterials.zip. In this working conversation, StudyGuide.txt was attached and the workshop notebooks were reviewed directly from their GitHub repositories. The prepared archive is available alongside this notebook for the group to upload if the instructor needs to see that exact step completed.

### Step 3 — Paste the professor's interview-preparation prompt

The prompt supplied in the workshop asks the LLM to review the ZIP and study guide, summarize them, compare topic coverage, conduct a 15-question interview quiz one question at a time, score the answers, and prepare personalized practice. The prompt is recorded below.

You are a seasoned Data Scientist, Machine Learning Engineer, and technical interviewer.
I am a Data Scientist and ML Engineer, fresh out of college. You will interview me for an ML Specialist role.

1) Unzip and read StudyMaterials.zip. Understand the workshop notebooks it contains. Produce a 500-word summary of the ML learning content and coding patterns.
2) Read StudyGuide.txt. Produce a 100-word summary of interview topics emphasized.
3) Match the study guide topics to the workshop materials. Create a table listing each topic, whether it is covered by the materials, and any gaps.
4) Create 15 multiple-choice questions (A-E) spanning supervised vs. unsupervised, variables, train/val/test, linear & logistic regression (R², MSE, cross-entropy), KNN (hyperparameters), decision trees (leaf nodes/predictions), plus scenario-based/behavioral items (e.g., imbalanced data, data leakage, model choice trade-offs). Ask one question at a time. After I answer all, score me.
5) Based on questions I get wrong, generate a new Jupyter Notebook named JobInterviewGuide_Workshop.ipynb inside a folder JobInterviewGuide_Workshop. Include clear Markdown explanations of weak topics, Python code scaffolding with exercises and TODOs, small realistic examples and sanity checks, and a short reflection prompt. Use the style and structure of the workshop notebooks in the ZIP as inspiration.

Stop here and wait for my command to start the quiz.

### Step 4 — Review the LLM's material summary and coverage table

The workshop notebooks show a practical machine learning workflow: define a prediction task; collect and inspect data; separate features from the target; split data into training and evaluation sets; preprocess carefully; fit a model; measure performance; interpret results; and document the work. Examples include predicting housing values, identifying flower species, classifying images, predicting pass/fail outcomes, and monitoring manufacturing robot readings. The notebooks use Python tools such as NumPy, pandas, Matplotlib, and scikit-learn, along with Jupyter and GitHub.

The regression workshops use linear regression to predict numeric values. The housing example uses median income to predict median house value. One notebook implements gradient descent from scratch and compares it with scikit-learn's LinearRegression. The model learns a slope and intercept to make predictions. Residuals show the differences between actual and predicted values. MSE, RMSE, MAE, and R² describe prediction error and explanatory power in different ways. Plots help identify relationships and possible limitations.

The logistic regression workshop changes the task to binary classification. A linear score passes through the sigmoid function to produce a probability between zero and one. A decision threshold turns the probability into a class. Cross-entropy, also called log loss, measures how well predicted probabilities match the correct labels and penalizes confident mistakes. The examples show why a probability score and a class label answer different questions.

The KNN workshop predicts a class using nearby training examples. The selected number of neighbours, distance measure, and feature scales affect the result. Standardization inside a pipeline helps distance-based methods compare features on a similar scale. Cross-validation can compare settings such as the number of neighbours. A decision tree instead asks a sequence of feature-based questions. Internal nodes split observations, while leaf nodes give the final prediction. Restricting tree depth can reduce overfitting.

The classification metrics workshop demonstrates that accuracy alone may hide poor results for a rare class. A confusion matrix separates true positives, true negatives, false positives, and false negatives. Precision describes how many predicted positives were correct; recall describes how many actual positives were detected; F1 combines these measures. The robot-monitoring material adds a broader data workflow by simulating readings, storing them, and displaying them for monitoring.

Across the notebooks, the key coding habits are using reproducible random states, inspecting data before modeling, keeping test data separate, fitting preprocessing only on training data, combining steps in pipelines, comparing models with suitable metrics, and documenting interpretations in Markdown. The main gaps are opportunities to practise unsupervised learning comparisons, residual-based linearization, tree leaf predictions, and selecting metrics based on real costs. Good interview answers should explain the choice, evidence, assumptions, and limitations in clear language.

Keep test data separate from tuning, because choosing settings from test results can inflate performance estimates. Stratified splits preserve class proportions, and pipelines fit preprocessing only during training. KNN relies on distances, so scaling its features matters; decision trees split at thresholds and usually need no scaling. Compare models on the same data with metrics suited to the task, then check data quality, class balance, and sample limits before sharing results.

#### Study guide summary

The study guide reviews machine learning foundations, Python tools, data preparation, scaling, regression, classification, optimization, and evaluation. It emphasizes features and targets, parameters and hyperparameters, generalization, overfitting, and model selection. Linear regression topics include slope, intercept, residuals, and error metrics. Logistic regression adds sigmoid probabilities, thresholds, and cross-entropy. KNN and decision trees introduce alternative prediction methods and complexity controls. Cross-validation and pipelines support reliable evaluation without data leakage. Applied examples include housing, diabetes, image classification, and robot monitoring. Interviews should test technical understanding, practical coding choices, interpretation of results, reproducibility, and clear explanations of assumptions, limitations, and trade-offs to employers.

#### Coverage and gaps

| Topic | Coverage in workshop materials | Gap or practice needed |
|---|---|---|
| Supervised and unsupervised learning | Supervised regression and classification are shown. | Practise explaining how unlabeled grouping differs from prediction using known labels. |
| Features and target | Examples identify inputs and prediction targets. | Practise identifying them in new scenarios. |
| Train, validation, and test | Splits and cross-validation are used. | Explain the separate role of each split. |
| Data leakage and pipelines | Scaling within pipelines is demonstrated. | Explain why preprocessing is fitted on training data only. |
| Linear regression and metrics | Slope, intercept, residuals, MSE, RMSE, MAE, and R² are covered. | Practise interpreting residual patterns and metric values. |
| Logistic regression and cross-entropy | Sigmoid probabilities, thresholds, and log loss are covered. | Explain why confident incorrect probabilities receive larger penalties. |
| KNN and decision trees | KNN scaling and tuning are shown; tree practice is included in the professor's notebook. | Practise how K and tree depth affect fit and generalization. |
| Imbalanced classification | Confusion matrices, accuracy, precision, recall, and F1 are covered. | Choose metrics based on the cost of false alarms and missed positives. |

### Step 5 — Complete the 15-question interview quiz

I asked to begin after reviewing the summaries. The LLM asked one multiple-choice question at a time. My answers and results are recorded below.

| # | Topic and question | My answer | Result |
|---|---|---|---|
| 1 | Predicting house prices from known selling prices versus grouping customers without labels: identify each learning type. | C — house prices supervised; customer grouping unsupervised. | Correct |
| 2 | A model uses house size, bedrooms, and age to predict selling price. Identify the target. | D — selling price. | Correct |
| 3 | Explain how training, validation, and test data should be used when comparing and tuning models. | A — train on training data, tune with validation, final evaluation on test. | Correct |
| 4 | Why can fitting a scaler on all rows before splitting cause data leakage? | B — test information influences preprocessing used for training. | Correct |
| 5 | Actual score 72, predicted score 80; residual is actual minus predicted. | C — -8. | Correct |
| 6 | Compare MSE 12 and MSE 20 on the same test data. | B — MSE 12 has smaller squared errors. | Correct |
| 7 | Interpret R² = 0.80. | C — about 80% of target variation is explained relative to a mean-prediction baseline. | Correct |
| 8 | What does the logistic regression sigmoid step do? | A — maps the score to a probability from 0 to 1. | Correct |
| 9 | For a true class of 1, which probability has a larger cross-entropy penalty? | D — probability 0.10 for class 1. | Correct |
| 10 | Why scale income and purchase-count features for KNN? | A — distance calculations could otherwise be dominated by the larger income values. | Correct |
| 11 | What is a likely risk of KNN with k = 1? | A — it can be sensitive to noise in one nearest example. | Correct |
| 12 | What happens at a decision-tree leaf node? | A — the model gives its final prediction. | Correct |
| 13 | A tree scores well on training data but poorly on test data. What change may reduce overfitting? | B — decrease max_depth. | Correct |
| 14 | A rare failure classifier has 98% accuracy but misses most failures. What should be examined first? | D — R². | Incorrect; correct answer: B — inspect recall and the confusion matrix, then consider the costs of missed failures. R² is for regression. |
| 15 | The model's test score is lower than its validation score. What should be reported and investigated? | B — report both, explain test estimates unseen performance, investigate without tuning directly on test. | Correct |

#### Quiz result

- Correct: 14 out of 15
- Score: 93.3%
- Topic to practise: choosing classification measures for imbalanced data, especially recall and confusion matrices
- Interview note: My spoken communication was not evaluated by this multiple-choice quiz.

### Step 6 — Generate practice for the missed topic

Because I missed the imbalanced-classification question, I received a focused practice notebook on why accuracy can be misleading, how to read a confusion matrix, and how lowering a classification threshold may affect recall and false alarms. The exercise includes a small TODO to calculate predictions and compare threshold settings.

### Step 7 — Complete the professor notebook's targeted coding exercises

I completed the three code exercises in the professor's JobInterviewGuide_Workshop notebook and added student-friendly interpretations.

1. **Train, validation, test, and leakage:** Used stratified splits and a scikit-learn pipeline. The scaler was fitted on training data only for the correct method. The leakage example fitted the scaler on the full dataset before splitting. The correct pipeline produced validation accuracy about 0.9882 and test accuracy about 0.9884. The leakage example used a different split size, so the printed scores are not a fair direct comparison.
2. **Linear and logistic regression:** Linear regression produced R² about 0.7850 and MSE about 210.0679. Its residual plot showed points generally scattered around zero without a clear curve. Logistic regression produced log loss about 0.3509. Its coefficients describe changes in log-odds for class 1, holding the other features constant.
3. **KNN and decision trees:** The Iris data was divided into 90 training, 30 validation, and 30 test examples. The best KNN validation setting was Euclidean distance with k = 5 and validation accuracy 1.0; final test accuracy was about 0.9667. The depth-3 decision tree scored 1.0 on the 30 test examples and had five leaves. The test set is small, so the score may not represent every future sample.

### Step 8 — Reflection

One of the most challenging topics for me was choosing the right way to evaluate a model when one class is much less common than the other. In the quiz, I first chose R² for a situation involving rare machine failures, but I learned that R² is used for regression, while a confusion matrix and recall help show how many real failures a classifier detects. I also learned that a model can have 98% accuracy and still be poor if it misses most of the failures, so we need to think about the cost of missed failures and false alarms. The exercises helped me understand why we use validation data to compare model settings and keep test data for a final check. Over the next week, I plan to practise reading confusion matrices, comparing precision and recall, and explaining which metric makes sense for a given problem.

### Step 9 — Submission items for my contribution

- Add my completed results and reflection to the professor's workshop notebook.
- Keep this section in JobInterviewLLMSession.ipynb as the record of my LLM process.
- Include the StudyMaterials.zip archive when the team assembles the materials.
- The team PDF includes our course, group name, member names and IDs, and project repository link.
